# News Articles Grouping Research

The central hypothesis of this research is that by combining multiple layers of processing and analysis—rather than relying solely on semantic or topical similarity—we can more accurately group news articles into meaningful real-world events.

In other words, we do not want to cluster articles together simply because they mention the same high-level entity (e.g., the President of Argentina). Instead, we aim to group them based on the specific event that occurred. For example:

- *The Argentine president approves a set of laws preventing X, Y, and Z.*
- *The Argentine president vetoes a tax reform proposal.*

Although both articles mention the same public figure, they describe different real-world events and therefore should belong to different clusters.

---

## Baseline

As a baseline, we first preprocess the articles by generating embeddings using `jina-embedding-v3`. Its ~8k token context window allows us to embed long news articles directly. In cases where an article exceeds the maximum context length, we chunk it and compute the average of its embeddings to obtain a single vector representation.

After embedding, we apply **HDBSCAN**, a density-based clustering algorithm that identifies clusters of varying density and can label outliers as noise. HDBSCAN does not require specifying the number of clusters in advance, making it suitable for exploratory event grouping.

While the algorithm performs well for certain clearly separated events, in many cases it clusters articles based on broad semantic similarity or shared topics rather than the specific real-world event being described. This confirms the limitation of relying solely on embedding-based semantic clustering.

*[Table with evaluation metrics — to be calculated]*

---

## Research Structure

### 1. Stage 1 - Article Pair Generation

- **Article preprocessing:**  
  We embed all articles using `jina-embedding-v3` with a 768-dimensional representation. This provides a dense numerical representation of each article.

- **Similarity search algorithms:**  
  We compare two common nearest-neighbor search approaches:

  - **k-Nearest Neighbors (KNN):**  
    Focuses on exact similarity search, prioritizing precision at the expense of computational speed.

  - **Approximate Nearest Neighbors (ANN):**  
    Trades a small amount of accuracy for significantly improved speed and scalability, retrieving results that are "close enough."

  We will evaluate both approaches to determine which provides the best trade-off between efficiency and retrieval quality for our pipeline.

- **Pair construction:**  
  For each article, we retrieve its top *k* most similar articles (e.g., 10–20). We then construct pairs of the form:  
  (A, A₁), (A, A₂), ..., (A, Aₖ).  
  These candidate pairs will serve as inputs for deeper event-level comparison.

<br/>

### 2. Stage 2 - Pair Comparison

The objective of this stage is to determine whether two articles within a candidate pair describe the same real-world event. We use two complementary methods:

- **Cross-Encoder Classification Model:**  
  A custom fine-tuned transformer that takes both articles jointly as input and outputs a probability indicating whether they refer to the same event. Unlike bi-encoder embeddings, a cross-encoder evaluates the interaction between the two texts directly, enabling finer-grained comparison.

- **Entity Extraction and Analysis:**  
  This step includes:
  1. Named Entity Recognition (NER) to extract entities.
  2. Entity normalization using a knowledge base (Wikidata) to resolve aliases and canonicalize references.
  3. Jaccard similarity between the two normalized entity sets to compute an entity-overlap score.

Each method produces a similarity score. We then compute a weighted combination of these scores to obtain a final event-level similarity score for each article pair.

<br/>

### 3. Graph Generation

Once similarity scores are computed for all candidate pairs, we construct a graph:

- Each node represents an article.
- An edge between two nodes represents their final similarity score.
- The graph is sparse, as each article is only connected to its top-*k* nearest neighbors.
- Weak edges (below a predefined threshold) are removed to reduce noise.

We then apply and compare two graph-based grouping algorithms:

- **Louvain:**  
  A community detection algorithm that optimizes modularity and is well-suited for weighted graphs.

- **Connected Components:**  
  A simpler approach that groups nodes based solely on connectivity.

By comparing these methods, we aim to determine which produces more coherent event-level clusters for our task. The best-performing configuration will be adopted in the final research pipeline.

### Import Dependencies

In [1]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from tqdm import tqdm

In [2]:
DATASET_PATH = "/content/drive/MyDrive/Research/articles_grouping/articles_events.csv"
EMBEDDINGS_OUTPUT_PATH = "/content/drive/MyDrive/Research/articles_grouping/embeddings.dat"

In [3]:
MODEL_NAME = "intfloat/multilingual-e5-large"

## 1. Data Preparation

In this section, we generate embeddings for approximately ~500k news articles using a combination of each article's **title and content** as input. This provides a richer contextual representation than using either field independently.

For embedding generation, we use the `sentence_transformers` library within a Google Colab environment. If the experiment is replicated locally, the same process can be performed using `Ollama`, provided the model is available and properly configured.

As introduced earlier, we use `jina-embeddings-v3`, which supports multilingual embeddings (including English and Spanish) and allows context windows of up to 8,192 tokens. This makes it well-suited for long-form news articles.

To ensure efficient memory usage, we implement a batching strategy during embedding generation. The resulting embeddings are stored on disk, allowing us to compute them only once and reuse them across multiple experiments without recomputation.

In [4]:
# Setup Google Drive in Colab
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


#### Load Dataset

All articles should have values based on the data analysis notebook results but just in case we add the extra check.

In [5]:
df = pd.read_csv(filepath_or_buffer=DATASET_PATH)
df = df.reset_index(drop=True)

df["title"] = df["title"].fillna("")
df["content"] = df["content"].fillna("")
df["text"] = "passage: " + df["title"] + "\n" + df["content"] # How the embedder needs to receive the corpus

print(f"Dataset loaded and formatted successfully with {df.shape[0]} examples")

Dataset loaded and formatted successfully with 647442 examples


#### Load Embedder Model

In [ ]:
emb_model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

In [8]:
TOTAL = len(df)
EMBEDDING_DIM = 1024
BATCH_SIZE = 2048

embeddings_mmap = np.memmap(EMBEDDINGS_OUTPUT_PATH, dtype="float32", mode="w+", shape=(TOTAL, EMBEDDING_DIM))

for i in tqdm(range(0, TOTAL, BATCH_SIZE)):
    batch = df["text"].iloc[i : i + BATCH_SIZE].tolist()
    embeddings_mmap[i : i + BATCH_SIZE] = emb_model.encode(
        batch,
        batch_size=256,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    if i % (BATCH_SIZE * 10) == 0:
        embeddings_mmap.flush()

del embeddings_mmap

100%|██████████| 317/317 [2:54:51<00:00, 33.10s/it]


In [19]:
# Verify the file was written correctly
embeddings = np.memmap(EMBEDDINGS_OUTPUT_PATH, dtype="float32", mode="r", shape=(TOTAL, EMBEDDING_DIM))

print(f"Shape: {embeddings.shape}")        # should be (647442, 1024)
print(f"First embedding: {embeddings[0][:5]}")  # should be non-zero values
print(f"Last embedding: {embeddings[-1][:5]}")  # should be non-zero values

Shape: (647442, 1024)
First embedding: [ 0.028555   -0.01379716 -0.01848934 -0.01493137  0.03227536]
Last embedding: [ 0.05467785 -0.01353509 -0.03169908 -0.03014548  0.02629345]
